# Notebook 15 — TTLinear と Dense `nn.Linear` の Forward 等価性検証

## 目的

この Notebook の目的は、圧縮率評価・rank truncation・fine-tuning・Fashion-MNIST 学習ではありません。

確認するのは次の1点だけです。

$$
\boxed{
W_{\mathrm{TT}} = W
\quad\Longrightarrow\quad
XW_{\mathrm{TT}}^{\mathsf T}+b
=
XW^{\mathsf T}+b
}
$$

PyTorch の dense `nn.Linear` と、TT-matrix core による **dense weight を復元しない direct contraction** が、truncation なしでは同じ線形写像を計算することを、

- 数式
- 添字
- shape
- 小さい PyTorch 実験

で確認します。

今回は 2-core TT-matrix に固定します。

数学の添字は 1 始まりで書きます。Python / PyTorch の tensor index は 0 始まりなので、コード中ではその違いに注意します。


## 1. 実行環境と再現性

すべて CPU + `torch.float64` で実行します。

乱数 seed は `0` に固定します。


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import prod

# 再現性を固定
torch.manual_seed(0)

# 今回は float64 を標準にする
torch.set_default_dtype(torch.float64)

# CPU 固定
device = torch.device("cpu")

print("PyTorch version:", torch.__version__)
print("default dtype:", torch.get_default_dtype())
print("device:", device)


PyTorch version: 2.11.0+cu128
default dtype: torch.float64
device: cpu


## 2. Dense `nn.Linear` の向き

PyTorch の `nn.Linear(in_features=n, out_features=m)` は、重みを

$$
W\in\mathbb R^{m\times n}
$$

として保持します。

今回の小例では、

$$
B=2,\qquad n=6,\qquad m=8.
$$

入力、重み、bias、出力は、

$$
X\in\mathbb R^{B\times n},
\qquad
W\in\mathbb R^{m\times n},
\qquad
b\in\mathbb R^m,
$$

$$
Y\in\mathbb R^{B\times m}.
$$

forward は、

$$
\boxed{
Y=XW^{\mathsf T}+b
}
$$

です。

数学の添字を 1 始まりにすると、

$$
\boxed{
Y_{\beta,i}
=
\sum_{j=1}^{n}
X_{\beta,j}W_{i,j}
+b_i
}
$$

です。

ここで、

- $\beta=1,\ldots,B$：batch index
- $i=1,\ldots,m$：出力 feature index
- $j=1,\ldots,n$：入力 feature index

です。

$W^{\mathsf T}$ が式に現れますが、成分式では $W_{i,j}$ になります。

$$
(XW^{\mathsf T})_{\beta,i}
=
\sum_{j=1}^{n}
X_{\beta,j}(W^{\mathsf T})_{j,i}
=
\sum_{j=1}^{n}
X_{\beta,j}W_{i,j}.
$$

したがって `linear.weight[i, j]` は、出力ユニット `i` と入力 feature `j` の結合係数です。


In [18]:
B = 2
n = 6
m = 8

# PyTorch の Linear weight は (out_features, in_features) = (m, n)
linear = nn.Linear(
    in_features=n,
    out_features=m,
    bias=True,
    dtype=torch.float64,
    device=device,
)

x = torch.randn(
    B,
    n,
    dtype=torch.float64,
    device=device,
)

# 3 通りで同じ dense Linear を計算
y_module = linear(x)
y_functional = F.linear(x, linear.weight, linear.bias)
y_manual = x @ linear.weight.T + linear.bias

print("x.shape:", tuple(x.shape))
print("weight.shape:", tuple(linear.weight.shape))
print("bias.shape:", tuple(linear.bias.shape))
print("y.shape:", tuple(y_module.shape))

assert x.shape == (B, n)
assert linear.weight.shape == (m, n)
assert linear.bias.shape == (m,)
assert y_module.shape == (B, m)

torch.testing.assert_close(
    y_module,
    y_functional,
    rtol=1e-12,
    atol=1e-12,
)

torch.testing.assert_close(
    y_module,
    y_manual,
    rtol=1e-12,
    atol=1e-12,
)

print("OK: nn.Linear == F.linear == x @ weight.T + bias")


x.shape: (2, 6)
weight.shape: (8, 6)
bias.shape: (8,)
y.shape: (2, 8)
OK: nn.Linear == F.linear == x @ weight.T + bias


### Dense forward の1要素を Python loop で確認する

数学では、

$$
Y_{\beta,i}
=
\sum_{j=1}^{n}
X_{\beta,j}W_{i,j}+b_i
$$

でした。

コードは 0 始まりなので、ここでは `beta = 1`, `i = 3` という Python index を選び、

```python
for j in range(n):
```

で入力 index を明示的に足します。


In [19]:
# Python / PyTorch は 0 始まり
beta = 1
i_out = 3

y_loop = linear.bias[i_out].clone()

for j_in in range(n):
    y_loop = y_loop + x[beta, j_in] * linear.weight[i_out, j_in]

print("loop result:", y_loop.item())
print("module result:", y_module[beta, i_out].item())
print(
    "abs error:",
    (y_loop - y_module[beta, i_out]).abs().item(),
)

torch.testing.assert_close(
    y_loop,
    y_module[beta, i_out],
    rtol=1e-12,
    atol=1e-12,
)

print("OK: 1要素の index 展開が nn.Linear と一致")


loop result: 1.363470070814456
module result: 1.363470070814456
abs error: 0.0
OK: 1要素の index 展開が nn.Linear と一致


## 3. `weight` の tensorization

今回は core 数を、

$$
d=2
$$

に固定します。

出力 mode を、

$$
(m_1,m_2)=(2,4),
$$

入力 mode を、

$$
(n_1,n_2)=(2,3)
$$

とします。

したがって、

$$
m=m_1m_2=2\cdot4=8,
$$

$$
n=n_1n_2=2\cdot3=6.
$$

元の重みは、

$$
W\in\mathbb R^{8\times6}
$$

です。

### Step 1：`reshape`

まず dense weight を、

$$
(m,n)
\rightarrow
(m_1,m_2,n_1,n_2)
$$

へ reshape します。

shape は、

$$
(8,6)
\rightarrow
(2,4,2,3).
$$

`reshape` は値そのものをシャッフルするのではなく、flat index の見方を複数の mode index に変えます。

### Step 2：site ごとの axis 順に並べ替える

TT-matrix の第1 site は $(m_1,n_1)$、第2 site は $(m_2,n_2)$ を担当します。

したがって axis 順を、

$$
(m_1,m_2,n_1,n_2)
$$

から、

$$
\boxed{
(m_1,n_1,m_2,n_2)
}
$$

へ変える必要があります。

### 演習1の目的

PyTorch の `reshape` と `permute` を使って、

```text
(8, 6)
→ (2, 4, 2, 3)
→ (2, 2, 4, 3)
```

を作ってください。

元の dense `(m,n)` に戻すときには、この axis permutation の逆変換が必要になります。


In [20]:
# TODO 1:
# linear.weight を TT-matrix 用の axis 順へ tensorize してください。
#
# 固定値:
# m1, m2 = 2, 4
# n1, n2 = 2, 3
#
# 確認する shape:
# linear.weight        : (8, 6)
# weight_4d            : (2, 4, 2, 3)
# weight_interleaved   : (2, 2, 4, 3)
#
# やること:
# 1. m1, m2, n1, n2 を定義
m1, m2 = 2, 4
n1, n2 = 2, 3
# 2. linear.weight を grouped order に reshape
weight_4d = linear.weight.reshape(m1, m2, n1, n2)          # (2, 4, 2, 3)
# 3. site ごとの (m_k, n_k) が隣り合うよう axis を並べ替える
weight_interleaved = weight_4d.permute(0, 2, 1, 3)  
# 4. shape を print / assert
print("linear.weight.shape:",weight_interleaved.shape)


linear.weight.shape: torch.Size([2, 2, 4, 3])


### 演習2 — `reshape` / `permute` 後も同じ要素を指しているか

数学の添字を 1 始まりで書くと、flat な行 index $i$ と列 index $j$ は、

$$
i=(i_1-1)m_2+i_2,
$$

$$
j=(j_1-1)n_2+j_2
$$

に対応します。

Python は 0 始まりなので、

$$
i=i_1m_2+i_2,
\qquad
j=j_1n_2+j_2
$$

です。

任意の $(i_1,i_2,j_1,j_2)$ を選び、

- 元の `linear.weight[i, j]`
- tensorization 後の同じ要素

が一致することを確認してください。

### 考えること

- `reshape` だけでは axis 順は $(m_1,m_2,n_1,n_2)$ のままであること
- TT-matrix の site は $(m_k,n_k)$ のペアを持つこと
- `permute` 後に、どの axis が $i_1,j_1,i_2,j_2$ に対応するか


In [21]:
# TODO 2:
# tensorization 前後で同じ weight 要素を参照できることを確認してください。
#
# 任意の 0-based multi-index
# (i1_idx, i2_idx, j1_idx, j2_idx)
# を選び、
#
# i_flat = i1_idx * m2 + i2_idx
# j_flat = j1_idx * n2 + j2_idx
#
# を使って dense 側と interleaved 側を比較してください。
#
# torch.testing.assert_close を使って確認してください。
#
# --- 設定 ---
# 直前セルの linear / weight_interleaved / m1, m2, n1, n2 を使う
i1_idx, i2_idx = 1, 2
j1_idx, j2_idx = 0, 1

i_flat = i1_idx * m2 + i2_idx
j_flat = j1_idx * n2 + j2_idx

# weight_interleaved: (m1, n1, m2, n2) → [i1, j1, i2, j2]
# linear.weight:      (m, n)           → [i_flat, j_flat]
w_dense = linear.weight[i_flat, j_flat]
w_tt = weight_interleaved[i1_idx, j1_idx, i2_idx, j2_idx]

print("W[i_flat, j_flat] =", w_dense.item())
print("A[i1, j1, i2, j2] =", w_tt.item())

torch.testing.assert_close(w_dense, w_tt)
print("OK")


W[i_flat, j_flat] = -0.015652710424751803
A[i1, j1, i2, j2] = -0.015652710424751803
OK


## 4. 演習3 — Rank truncation なしの 2-core TT-SVD

`weight_interleaved` の shape は、

$$
(m_1,n_1,m_2,n_2)
$$

です。

第1 site と第2 site の間で unfolding すると、

$$
A
\in
\mathbb R^{
(m_1n_1)\times(m_2n_2)
}
$$

です。

今回、

$$
m_1n_1=2\cdot2=4,
$$

$$
m_2n_2=4\cdot3=12,
$$

なので、

$$
A\in\mathbb R^{4\times12}.
$$

SVD は、

$$
A=U\Sigma V^{\mathsf T}
$$

です。

今回は **rank truncation を一切行いません**。

保持 rank は、

$$
r_1=\text{number of singular values}
$$

です。

gauge は、

$$
G^{(1)}=\operatorname{reshape}(U),
$$

$$
G^{(2)}
=
\operatorname{reshape}(\Sigma V^{\mathsf T})
$$

とします。

core shape は、

$$
G^{(1)}
\in
\mathbb R^{1\times m_1\times n_1\times r_1},
$$

$$
G^{(2)}
\in
\mathbb R^{r_1\times m_2\times n_2\times1}.
$$

### TODO

- unfolding $A$ を作る
- `torch.linalg.svd(..., full_matrices=False)` を実行する
- truncationせず全特異値を保持する
- `core1`, `core2` を作る
- shape を assert する
### 境界 TT rank の添字について

境界 TT rank は、

$$
r_0=r_2=1
$$

です。

数学では、その長さ 1 の境界軸の唯一の添字を、

$$
\alpha_0=\alpha_2=1
$$

と書きます。

一方、Python / PyTorch は 0 始まりなので、同じ長さ 1 の境界軸の唯一の index は `0` です。

したがって数学では、

$$
G^{(1)}_{1,i_1,j_1,\alpha_1},
\qquad
G^{(2)}_{\alpha_1,i_2,j_2,1}
$$

と書き、コードではそれぞれ `core1[0, ...]`、`core2[..., 0]` と対応します。


In [22]:
# TODO 9:
# dense_to_tt_matrix_tensor を実装してください。
#
from math import prod
import torch
def dense_to_tt_matrix_tensor(
    W: torch.Tensor,
    out_modes: list[int] | tuple[int, ...],
    in_modes: list[int] | tuple[int, ...],
    *,
    verbose: bool = False,
) -> torch.Tensor:
    """dense weight W を TT-SVD 入力用の combined tensor へ並べ替える。

    処理の流れ:
        1. W.shape == (m, n) を検証（m,n = product(out/in_modes)）
        2. grouped: (m_1,...,m_d, n_1,...,n_d) へ reshape
        3. grouped → interleaved: (m_1,n_1,...,m_d,n_d) へ permute
        4. 各 site で (m_k, n_k) → q_k = m_k * n_k に combine
        5. shape (q_1, ..., q_d) を返す

    引数:
        W: dense weight, shape (m, n)
            dtype float64, device CPU（入力は変更しない）
        out_modes: 行側 modes (m_1, ..., m_d)
        in_modes: 列側 modes (n_1, ..., n_d)
        verbose: True なら grouped / perm / interleaved / combined を表示

    戻り値:
        A: combined tensor, shape (q_1, ..., q_d)
            q_k = m_k * n_k

    注意:
        TT-SVD 自体はこの関数では行わない（並べ替えのみ）。
    """

    # list / tuple の違いをここでなくす。
    # 以後は shape と比較しやすい tuple として扱う。
    out_modes = tuple(out_modes)
    in_modes = tuple(in_modes)

    # =========================================================
    # 1. validate
    # =========================================================

    # site 数 d を決める。
    # 例:
    # out_modes = (2, 3, 4) なら d = 3
    d = len(out_modes)

    # W は行列である必要がある。
    assert W.ndim == 2,(
        print(f"W must be 2D, got shape={tuple(W.shape)}")
    )
    assert len(out_modes) == len(in_modes)
    assert len(out_modes) > 0

    assert all(mode > 0 for mode in out_modes)
    assert all(mode > 0 for mode in in_modes)

    assert W.dtype == torch.float64
    # assert W.device.type == "cpu"
    # 出力側と入力側で site 数が一致する必要がある。
    # 例:
    # out_modes = (2, 3, 4)
    # in_modes  = (5, 6, 7)
    # → OK
    #
    # out_modes = (2, 3)
    # in_modes  = (4, 5, 6)
    # → NG
    assert len(out_modes) == len(in_modes),(
        print(f"len(out_modes)={len(out_modes)}, len(in_modes)={len(in_modes)}")
    )

    # 空の mode list は許可しない。
    # d=0 では TT-SVD 用 tensor を作る意味がない。
    assert d != 0,(
        print("d=0")
    )

    # 各 mode は正の整数である必要がある。
    # 例:
    # (2, 3, 4) → OK
    # (2, 0, 4) → NG
    # (2, -3, 4) → NG
    assert all(mode > 0 for mode in out_modes), (
        f"out_modes の各要素は正の整数である必要があります: {out_modes}"
    )
    assert all(mode > 0 for mode in in_modes), (
        f"in_modes の各要素は正の整数である必要があります: {in_modes}"
    )

    # dense W の実際の shape を取得する。
    # m: output feature 数
    # n: input feature 数
    m, n = W.shape

    # modes が要求する dense shape を作る。
    # expected_m = m1 * m2 * ... * md
    # expected_n = n1 * n2 * ... * nd
    expected_m = 1
    for mode in out_modes:
        expected_m = mode * expected_m
    expected_n = 1
    for mode in in_modes:
        expected_n = mode * expected_n

    # W.shape が modes と一致するかを確認する。
    # 例:
    # W.shape = (6, 20)
    # out_modes = (2, 3)  -> expected_m = 6
    # in_modes  = (4, 5)  -> expected_n = 20
    # → OK
    # W.shape = (7, 20) なら NG
    assert expected_m == m,(
        print(f"miss match shape,expected_m={expected_m },m={m}")
    )
    assert expected_n == n,(
        print(f"miss match shape,expected_n={expected_n },m={n}")
    ) 

    # 2. grouped
    # W の row axis を out_modes に、
    # W の column axis を in_modes に分解する reshape を書く。
    grouped = W.reshape(*out_modes , *in_modes)

    # 3. interleaved
    # grouped の axis:
    # (m1, ..., md, n1, ..., nd)
    #
    # 欲しい axis:
    # (m1, n1, ..., md, nd)
    #
    # perm を loop で構築する。
    perm = []

    for k in range(d):
        # output mode m_{k+1} の元 axis を追加
        perm.append(k)
        # input mode n_{k+1} の元 axis を追加
        perm.append(d + k)

    interleaved = grouped.permute(*perm)

    # 4. combined
    # q_k = m_k * n_k を site ごとに作る。
    q_modes = tuple(
        m_k * n_k
        for m_k, n_k in zip(out_modes, in_modes)
    )

    # (m1, n1, ..., md, nd)
    # -> (q1, ..., qd)
    return interleaved.reshape(*q_modes)


In [23]:
from collections.abc import Sequence

def tt_cores_to_tt_matrix_cores(
    cores: Sequence[torch.Tensor],
    out_modes: Sequence[int],
    in_modes: Sequence[int],
) -> list[torch.Tensor]:
    """通常 TT cores を TT-matrix cores へ reshape する。

    各 core の物理軸 q_k = m_k * n_k を (m_k, n_k) にばらす。

        core_k: (r_{k-1}, q_k, r_k)
            → (r_{k-1}, m_k, n_k, r_k)

    引数:
        cores: 通常 TT cores
            各要素 shape (r_left, q_k, r_right)
            q_k = out_modes[k] * in_modes[k]
        out_modes: 行側 modes (m_1, ..., m_d)
        in_modes: 列側 modes (n_1, ..., n_d)

    戻り値:
        TT-matrix cores
            各要素 shape (r_left, m_k, n_k, r_right)

    注意:
        入力 cores は変更しない（新しい list / tensor を返す）。
        TT-SVD 自体は行わない（reshape のみ）。
    """
    out_modes = tuple(out_modes)
    in_modes = tuple(in_modes)

    assert len(cores) >= 1, "cores は1個以上必要です"
    assert len(out_modes) == len(in_modes) == len(cores), (
        f"len(cores)={len(cores)}, len(out_modes)={len(out_modes)}, "
        f"len(in_modes)={len(in_modes)}"
    )

    tt_matrix_cores: list[torch.Tensor] = []
    for core, m_k, n_k in zip(cores, out_modes, in_modes):
        assert core.ndim == 3, (
            f"通常 TT core は3階である必要があります: got ndim={core.ndim}, shape={tuple(core.shape)}"
        )
        r_left, q_k, r_right = core.shape
        assert q_k == m_k * n_k, (
            f"q_k={q_k} が m_k*n_k={m_k * n_k} と一致しません "
            f"(m_k={m_k}, n_k={n_k})"
        )
        tt_matrix_cores.append(core.reshape(r_left, m_k, n_k, r_right))

    return tt_matrix_cores


In [24]:
from collections.abc import Sequence

from nn_compression.compression import tt_svd, tt_svd_exact

def dense_to_tt_matrix_cores(
    W: torch.Tensor,
    out_modes: Sequence[int],
    in_modes: Sequence[int],
    *,
    max_rank: int | None = None,  # None なら truncation なし
) -> list[torch.Tensor]:
    """
    W (m, n)
      → combined A (q1,...,qd)     # dense_to_tt_matrix_tensor
      → TT cores (r, q, r)         # tt_svd / tt_svd_exact
      → TT-matrix cores (r, m, n, r)
    """
    tt_core = dense_to_tt_matrix_tensor(W, out_modes, in_modes)
    if max_rank is None:
        tt_cores_3d: list[torch.Tensor] = tt_svd_exact(tt_core)
    else:
        tt_cores_3d = tt_svd(tt_core, max_rank)
    tt_cores_4d: list[torch.Tensor] = tt_cores_to_tt_matrix_cores(
        tt_cores_3d, out_modes, in_modes
    )
    return tt_cores_4d
    
    


In [25]:
# TODO 3:
# truncation なしの 2-core TT-SVD を実装してください。
#
# 作るもの:
# A
# U, S, Vh
# r1
# core1
# core2
#
# 期待 shape:
# A      : (m1 * n1, m2 * n2)
# core1  : (1, m1, n1, r1)
# core2  : (r1, m2, n2, 1)
#
# full_matrices=False を使ってください。
#
# --- 設定 ---
# 直前セルの weight_interleaved / m1, m2, n1, n2 と
# tt_cores_to_tt_matrix_cores を使う
out_modes = (m1, m2)
in_modes = (n1, n2)

# 1. site1 | site2 の unfolding
A = weight_interleaved.reshape(m1 * n1, m2 * n2)

# 2. truncation なし SVD
U, S, Vh = torch.linalg.svd(A, full_matrices=False)
r1 = int(S.shape[0])

# 3. 通常 TT cores (r, q, r) を作り、TT-matrix cores へ変換
cores_3d = [
    U.reshape(1, m1 * n1, r1),
    (S[:, None] * Vh).reshape(r1, m2 * n2, 1),
]
core1, core2 = tt_cores_to_tt_matrix_cores(cores_3d, out_modes, in_modes)

print("A.shape:", tuple(A.shape))
print("U,S,Vh:", tuple(U.shape), tuple(S.shape), tuple(Vh.shape))
print("r1:", r1)
print("core1.shape:", tuple(core1.shape))
print("core2.shape:", tuple(core2.shape))

assert A.shape == (m1 * n1, m2 * n2)
assert core1.shape == (1, m1, n1, r1)
assert core2.shape == (r1, m2, n2, 1)
print("OK: 2-core TT-SVD (no truncation)")

# ラッパ経路とも一致することを確認
cores_via_helper = dense_to_tt_matrix_cores(
    linear.weight.detach(),
    out_modes,
    in_modes,
    max_rank=None,
)
torch.testing.assert_close(core1, cores_via_helper[0], rtol=1e-12, atol=1e-12)
torch.testing.assert_close(core2, cores_via_helper[1], rtol=1e-12, atol=1e-12)
print("OK: hand SVD ≈ dense_to_tt_matrix_cores(max_rank=None)")


A.shape: (4, 12)
U,S,Vh: (4, 4) (4,) (4, 12)
r1: 4
core1.shape: (1, 2, 2, 4)
core2.shape: (4, 4, 3, 1)
OK: 2-core TT-SVD (no truncation)
OK: hand SVD ≈ dense_to_tt_matrix_cores(max_rank=None)


## 5. 演習4 — Core から dense weight を復元する

2-core TT-matrix の要素式は、

$$
\boxed{
\mathcal W_{i_1,j_1,i_2,j_2}
=
\sum_{\alpha_1=1}^{r_1}
G^{(1)}_{1,i_1,j_1,\alpha_1}
G^{(2)}_{\alpha_1,i_2,j_2,1}
}
$$

です。

まず `torch.einsum` を使い、

$$
(m_1,n_1,m_2,n_2)
$$

の interleaved weight を復元します。

縮約する添字は、

- 第1 core の右 bond $\alpha_1$
- 第2 core の左 bond $\alpha_1$
- 両端の rank-1 boundary

です。

その後、

$$
(m_1,n_1,m_2,n_2)
\rightarrow
(m_1,m_2,n_1,n_2)
$$

へ inverse permutation し、

$$
(m,n)
$$

へ reshape します。

### TODO

1. `einsum` の添字を自分で決める
2. interleaved weight を復元
3. inverse permutation
4. `(m,n)` へ reshape
5. `linear.weight` と `assert_close`

### 考えること

出力 axis を必ず、

$$
(i_1,j_1,i_2,j_2)
$$

の順にするには、`einsum` の output 添字をどう並べればよいでしょうか。


In [26]:
# TODO 4:
# core1 / core2 から dense weight を復元してください。
#
# 1. torch.einsum で
#    (m1, n1, m2, n2)
#    の interleaved tensor を作る
print(core1.shape)
print(core2.shape)
dense_core = torch.einsum("abcd,defg->bcef",core1,core2)
#
# 2. inverse permutation で
#    (m1, m2, n1, n2)
#    に戻す
dense_core = dense_core.permute(0,2,1,3)
# 3. (m, n) に reshape
dense_core = dense_core.reshape(m1*m2,n1*n2)
#
# 4. linear.weight と torch.testing.assert_close
torch.testing.assert_close(
    dense_core,
    linear.weight.detach(),
    rtol=1e-12,
    atol=1e-12,
)
print("OK: reconstructed weight ≈ linear.weight")

# 注意:
# 完成した einsum 文字列はここには書いていません。


torch.Size([1, 2, 2, 4])
torch.Size([4, 4, 3, 1])
OK: reconstructed weight ≈ linear.weight


### 演習5 — Weight の1要素を bond index の Python loop で確認する

任意の flat index $(i,j)$ を、

$$
i\leftrightarrow(i_1,i_2),
\qquad
j\leftrightarrow(j_1,j_2)
$$

へ変換し、

$$
\boxed{
W_{i,j}^{\mathrm{TT}}
=
\sum_{\alpha_1=1}^{r_1}
G^{(1)}_{1,i_1,j_1,\alpha_1}
G^{(2)}_{\alpha_1,i_2,j_2,1}
}
$$

を Python loop で直接計算します。

### TODO

- `divmod` で flat index を multi-index にする
- $\alpha_1$ を loop する
- `weight_reconstructed[i, j]` と照合する


In [ ]:
# TODO 5:
# 任意の (i_flat, j_flat) について、
# TT bond rank alpha を明示的に loop して
# 1つの weight 要素を再構成してください。
#
# 確認:
# weight_loop ≈ weight_reconstructed[i_flat, j_flat]
#
# --- 設定 ---
# 直前セルの core1 / core2 / r1 / m1, m2, n1, n2 / dense_core を使う
weight_reconstructed = dense_core
i_flat, j_flat = 5, 2
i1_idx,i2_idx = divmod(i_flat,m2)
j1_idx,j2_idx = divmod(j_flat,n2)

weight_loop = torch.zeros((), dtype=core1.dtype, device=core1.device)

for alpha in range(r1):
    weight_loop = weight_loop + (
        core1[0, i1_idx, j1_idx, alpha] * core2[alpha, i2_idx, j2_idx, 0]
    )
print("weight_loop:",weight_loop)
print("weight_reconstructed:",weight_reconstructed[i_flat,j_flat])
torch.testing.assert_close(
    weight_loop,
    weight_reconstructed[i_flat, j_flat],
    rtol=1e-12,
    atol=1e-12,
)
print("OK: bond index の明示的な和が dense weight 要素と一致")



weight_loop: tensor(-0.2483, grad_fn=<AddBackward0>)
weight_reconstructed: tensor(-0.2483, grad_fn=<SelectBackward0>)


## 6. 演習6 — Dense weight を作らない Direct Contraction

ここが今回の中心です。

入力は、

$$
X\in\mathbb R^{B\times n}
$$

です。

まず、

$$
n=n_1n_2
$$

なので、

$$
\mathcal X
\in
\mathbb R^{B\times n_1\times n_2}
$$

へ reshape します。

### 右端 core から縮約する

core の数学的な列順は常に、

$$
G^{(1)}G^{(2)}
$$

です。

右側から縮約するというのは、core の順序を逆にすることではなく、**同じ総和をどの順番で評価するか**を選んでいるだけです。

最初に、

$$
\boxed{
T_{\beta,j_1,\alpha_1,i_2}
=
\sum_{j_2=1}^{n_2}
\mathcal X_{\beta,j_1,j_2}
G^{(2)}_{\alpha_1,i_2,j_2,1}
}
$$

を作ります。

shape は、

$$
(B,n_1,n_2)
\rightarrow
(B,n_1,r_1,m_2).
$$

次に、

$$
\boxed{
\mathcal Z_{\beta,i_1,i_2}
=
\sum_{j_1=1}^{n_1}
\sum_{\alpha_1=1}^{r_1}
T_{\beta,j_1,\alpha_1,i_2}
G^{(1)}_{1,i_1,j_1,\alpha_1}
}
$$

です。

学習のため、2回目の `einsum` の出力をまず、

$$
(B,m_2,m_1)
$$

の順に作ります。

そのまま flatten すると、

$$
i=i_1m_2+i_2
$$

という dense weight の row-major 規約と一致しません。

したがって、

$$
(B,m_2,m_1)
\rightarrow
(B,m_1,m_2)
$$

へ axis を直してから `(B,m)` に flatten します。

最後に bias を加えます。


### `einsum` 添字を自分で組み立てる

境界 rank はどちらも size 1 なので、教材としては先に、

```python
core2_no_boundary = core2[..., 0]
core1_no_boundary = core1[0]
```

として boundary 軸を落としてから考える方が分かりやすいです。

このとき shape は、

```text
x_tensor          : (B, n1, n2)
core2_no_boundary : (r1, m2, n2)
core1_no_boundary : (m1, n1, r1)
```

です。

#### 1回目

縮約するのは $j_2$ です。

欲しい出力は、

$$
T_{\beta,j_1,\alpha_1,i_2}
=
\sum_{j_2}
\mathcal X_{\beta,j_1,j_2}
G^{(2)}_{\alpha_1,i_2,j_2,1}
$$

なので、shape は、

```text
(B, n1, r1, m2)
```

です。

添字の意味は、

- batch index
- $j_1$
- $j_2$
- TT rank $\alpha_1$
- $i_2$

を区別して考えてください。

#### 2回目

次に、

$$
\mathcal Z_{\beta,i_1,i_2}
=
\sum_{j_1,\alpha_1}
T_{\beta,j_1,\alpha_1,i_2}
G^{(1)}_{1,i_1,j_1,\alpha_1}
$$

を計算します。

縮約するのは、

- $j_1$
- $\alpha_1$

です。

まず意図的に、

```text
(B, m2, m1)
```

の順で出力し、その後、

```text
(B, m2, m1)
→ (B, m1, m2)
→ (B, m)
```

と並べ直します。

### TODO

上の数式・shape・添字対応を手掛かりに、2本の `torch.einsum` を自分で書いてください。

**完成した `einsum` 文字列は、この Notebook にはあらかじめ書きません。**


In [ ]:
# TODO 6:
# dense weight を作らない direct contraction を実装してください。
#
# まず boundary rank 軸を落として考えてよいです。
#
# --- 設定 ---
# 直前セルの x / core1 / core2 / linear.bias / n1, n2, m1, m2 を使う
core2_no_boundary = core2[..., 0]  # (r1, m2, n2)
core1_no_boundary = core1[0]       # (m1, n1, r1)
#
# 1. x -> (B, n1, n2)
x = x.reshape(B, n1, n2)
#
# 2. core2_no_boundary と j2 を縮約
#    期待 shape: (B, n1, r1, m2)
tt2 = torch.einsum("abc,dec->abde",x , core2_no_boundary)
#
# 3. core1_no_boundary と j1 / r1 を縮約
#    期待 shape: (B, m2, m1)
# tt2: (B, n1, r1, m2), core1: (m1, n1, r1)
tt1 = torch.einsum("abcd,ebc->ade", tt2, core1_no_boundary)
#
# 4. axis を (B, m1, m2) に直す
tt = tt1.permute(0,2,1) 
#
# 5. (B, m) に reshape
tt = tt.reshape(B,-1) 
#
# 6. linear.bias を加えて y_tt を作る
y_tt = tt + linear.bias
print("y_tt.shape:", tuple(y_tt.shape))
#
# torch.einsum の添字は自分で組み立ててください。

y_tt.shape: (2, 8)


## 7. 演習7 — Direct Contraction の Correctness 検証

等価性を3段階に分けて確認します。

### 段階1：Dense Linear の実装同士

$$
\operatorname{nn.Linear}(X)
\approx
\operatorname{F.linear}(X,W,b)
$$

これは前半で確認済みです。

### 段階2：Weight の再構成

$$
W_{\mathrm{TT}}
\approx
W
$$

### 段階3：Forward

$$
Y_{\mathrm{TT}}
\approx
Y_{\mathrm{dense}}
$$

truncation は行っていません。

ただし SVD、GEMM、TT contraction では演算順序が異なるため、bitwise equality ではなく、

```python
torch.testing.assert_close(
    ...,
    rtol=1e-12,
    atol=1e-12,
)
```

で確認します。

### TODO

- dense実装同士
- TT reconstruction vs dense weight
- TT direct contraction vs dense forward

をそれぞれ独立に確認してください。

最後に、

- `max_abs_error_weight`
- `max_abs_error_output`
- TT rank `r1`
- dense parameter count
- TT core parameter count

を表示してください。

parameter count は core shape 理解の補助であり、この Notebook では圧縮率評価には進みません。


In [29]:
# TODO 7:
# 3段階の correctness 検証を実装してください。
#

# --- 設定 ---
# 直前セルの x / linear / y_tt / dense_core / core1 / core2 / r1 / B / n を使う
import torch.nn.functional as F
# すべて:
# rtol=1e-12
# atol=1e-12

rtol = 1e-12
atol = 1e-12
x_flat = x.reshape(B, n)
weight_reconstructed = dense_core


# 1. y_module と y_functional
y_module = linear(x_flat)
y_functional = F.linear(x_flat, linear.weight, linear.bias)
y_error = torch.linalg.vector_norm(y_module - y_functional)
print("y_error (L2):", y_error.item())
torch.testing.assert_close(y_module, y_functional, rtol=rtol, atol=atol)
print("OK: y_module ≈ y_functional")

# 2. weight_reconstructed と linear.weight
weight_error = torch.linalg.vector_norm(weight_reconstructed - linear.weight)
print("weight_error (L2):", weight_error.item())
torch.testing.assert_close(
    weight_reconstructed, linear.weight, rtol=rtol, atol=atol
)
print("OK: weight_reconstructed ≈ linear.weight")

# 3. y_tt と y_module / y_functional
torch.testing.assert_close(y_tt, y_module, rtol=rtol, atol=atol)
torch.testing.assert_close(y_tt, y_functional, rtol=rtol, atol=atol)
print("OK: y_tt ≈ y_module ≈ y_functional")

# 最後に:
max_abs_error_weight = (weight_reconstructed - linear.weight.detach()).abs().max()
max_abs_error_output = (y_tt - y_module).abs().max()
print("max_abs_error_weight:", max_abs_error_weight.item())
print("max_abs_error_output:", max_abs_error_output.item())

print("r1:", r1)
print("dense parameter count:", linear.weight.numel())  # m * n
print("TT core parameter count:", core1.numel() + core2.numel())



y_error (L2): 0.0
OK: y_module ≈ y_functional
weight_error (L2): 1.0062667213961117e-15
OK: weight_reconstructed ≈ linear.weight
OK: y_tt ≈ y_module ≈ y_functional
max_abs_error_weight: 4.996003610813204e-16
max_abs_error_output: 1.1102230246251565e-15
r1: 4
dense parameter count: 48
TT core parameter count: 64


## 8. 最終まとめ

この Notebook で確認する等価性は、

$$
\boxed{
W_{\mathrm{TT}}=W
\Longrightarrow
XW_{\mathrm{TT}}^{\mathsf T}+b
=
XW^{\mathsf T}+b
}
$$

です。

truncation なし TT-SVD で、

$$
W_{\mathrm{TT}}\approx W
$$

を確認し、dense weight を forward 中に復元しない direct contraction でも、

$$
Y_{\mathrm{TT}}\approx Y_{\mathrm{dense}}
$$

となることを float64 で自分で数値検証します。

direct contraction の出力が dense Linear と一致しない場合は、まず次の3点を確認します。

1. weight tensorization 時の `permute` 順序
2. TT core の physical mode の対応（$m_k$ と $n_k$ の取り違え）
3. 最終 output mode の `permute` / flatten 順序
